In [2]:
import numpy as np
import tensorflow as tf
import keras_tuner as kt
from sklearn.model_selection import train_test_split
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
from tensorflow.keras import layers, models, optimizers


In [ ]:
# --- CONFIGURAZIONE ---
SEED = 123
tf.keras.utils.set_random_seed(SEED)

IMG_WIDTH, IMG_HEIGHT, IMG_CHANNELS = 224, 224, 3  #immagini RGB di dimensione 224x224
IMG_SHAPE = (IMG_WIDTH, IMG_HEIGHT, IMG_CHANNELS)

NUM_CLASSI = 5 # classifico 5 classi: Grano, Mais, Soia, Riso, Terreno vuoto
NUM_CAMPIONI = 200 # numero totale di immagini 
BATCH_SIZE = 8  #metto batch size piccolo per evitare problemi di memoria (ResNet50 è un modello pesante)
EPOCHS = 3

CLASS_NAMES = ["Grano", "Mais", "Soia", "Riso", "Terreno vuoto"]

AUTOTUNE = tf.data.AUTOTUNE

In [ ]:
def genera_dati_agricoli(num_campioni, img_shape, num_classi,seed=SEED):
    """
    Simula il dataset 
    Le immagini sono randomiche e le etichette sono one-hot encoded.
    Restituisce X (immagini RGB) e y (etichette one-hot).
    """
    print(f"[DATI] Generazione di {num_campioni} campioni simulati (Noise)...")
    rng = np.random.default_rng(seed)
    
    # X: Immagini simulate (valori random 0-1)
    # Nota: ResNet di solito gradisce preprocessing specifico, ma per la struttura del codice
    # i valori normalizzati 0-1 funzionano tecnicamente per il training flow.
    X = rng.integers(low=0,high=256,size=(num_campioni,*img_shape)).astype(np.float32)

    # Classi simulate: numeri da 0 a 4
    y_classi = rng.integers(low=0,high=num_classi,size=(num_campioni,))
    #print(y_classi)  # Mostra le etichette simulate
    # Conversione in one-hot encoding
    y = np.eye(num_classi,dtype=np.float32)[y_classi]
    
    return X, y

# Generazione dei dati
X, y = genera_dati_agricoli(NUM_CAMPIONI, IMG_SHAPE, NUM_CLASSI,SEED)
print(f"Shape Input: {X.shape}, Shape Labels: {y.shape}")
#print(X[:5])  # Mostra le prime 5 immagini simulate
#print(y[:5])  # Mostra le prime 5 etichette one-hot

[DATI] Generazione di 200 campioni simulati (Noise)...
Shape Input: (200, 224, 224, 3), Shape Labels: (200, 5)


In [ ]:
def split_dati(X, y):
    """
    Divido i dati in training, validation e test set.:
    
    70% training (140 campioni),  per aggiornare i pesi del modello
    15% validation (30 campioni), per controllare il modello durante l'addestramento 
    15% test (30 campioni), per valutare le prestazioni finali del modello (su dati mai usati prima)

    Stratifico in base alle classi per mantenere la distribuzione delle etichette.
    """
    y_classi = y.argmax(axis=1)

    X_train, X_temp, y_train, y_temp = train_test_split(
        X,
        y,
        test_size=0.30,
        random_state=SEED,
        stratify=y_classi
    )

    X_val, X_test, y_val, y_test = train_test_split(
        X_temp,
        y_temp,
        test_size=0.50,
        random_state=SEED,
        stratify=y_temp.argmax(axis=1)
    )

    return X_train, X_val, X_test, y_train, y_val, y_test

X_train, X_val, X_test, y_train, y_val, y_test = split_dati(X, y)

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val:  ", X_val.shape, "y_val:  ", y_val.shape)
print("X_test: ", X_test.shape, "y_test: ", y_test.shape)

X_train: (140, 224, 224, 3) y_train: (140, 5)
X_val:   (30, 224, 224, 3) y_val:   (30, 5)
X_test:  (30, 224, 224, 3) y_test:  (30, 5)


In [32]:
#------------------------------
# CREAZIONE DATASET TENSORFLOW
#------------------------------

def crea_dataset(X, y, batch_size, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((X, y))

    if shuffle:
        ds = ds.shuffle(buffer_size=len(X), seed=SEED)

    ds = ds.batch(batch_size)  #crea batch di dimensione batch_size
    ds = ds.prefetch(AUTOTUNE) #ottimizza il caricamento dei dati in memoria per evitare colli di bottiglia durante l'addestramento

    return ds

train_ds = crea_dataset(X_train, y_train, BATCH_SIZE, shuffle=True)
val_ds = crea_dataset(X_val, y_val, BATCH_SIZE, shuffle=False)
test_ds = crea_dataset(X_test, y_test, BATCH_SIZE, shuffle=False)


In [33]:
#------------------------------
# COSTRUZIONE DEL MODELLO RESNET50
#------------------------------

def costruisci_modello_resnet50(img_shape, num_classi, dropout_rate=0.3):
    """
    Costruisce un modello di classificazione immagini basato su ResNet50.
    """

    base_model = ResNet50(
        weights="imagenet",
        include_top=False,
        input_shape=img_shape
    )

    # Congelo la base ResNet50 (ResNet50 non viene riaddestrata. Uso solo la conoscenza già appresa da ImageNet)
    base_model.trainable = False

    inputs = layers.Input(shape=img_shape)

    # Preprocessing specifico ResNet50
    x = preprocess_input(inputs)

    # Backbone pre-addestrato
    x = base_model(x, training=False)

    # Costruzione testa finale personalizzata
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(dropout_rate)(x)
    outputs = layers.Dense(num_classi, activation="softmax")(x)

    model = models.Model(inputs=inputs, outputs=outputs)

    #compilazione
    model.compile(
        optimizer=optimizers.Adam(learning_rate=0.001),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    # loss="categorical_crossentropy" perchè le etichette sono one-hot encoded (y_train, y_val, y_test)
    # loss="sparse_categorical_crossentropy" se le etichette fossero interi (0,1,2,3,4)
    
    return model


model = costruisci_modello_resnet50(
    img_shape=IMG_SHAPE,
    num_classi=NUM_CLASSI,
    dropout_rate=0.3
)

In [29]:
# -----------------------------
# RIEPILOGO MODELLO
# -----------------------------

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item (GetItem)  │ (None, 224, 224)  │          0 │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_1          │ (None, 224, 224)  │          0 │ input_layer_1[0]… │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_2          │ (None, 224, 224)  │          0 │ input_layer_1[0]… │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stack (Stack)       │ (None, 224, 224,  │          0 │ get_item[0][0],   │
│                     │ 3)                │            │ get_item_1[0][0], │
│                     │                   │            │ get_item_2[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 224, 224,  │          0 │ stack[0][0]       │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resnet50            │ (None, 7, 7,      │ 23,587,712 │ add[0][0]         │
│ (Functional)        │ 2048)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 2048)      │          0 │ resnet50[0][0]    │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 2048)      │          0 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 5)         │     10,245 │ dropout[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 23,597,957 (90.02 MB)

 Trainable params: 10,245 (40.02 KB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [30]:
# -----------------------------
# ADDESTRAMENTO DI PROVA
# -----------------------------

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

Epoch 1/3
5/5 ━━━━━━━━━━━━━━━━━━━━ 34s 4s/step - accuracy: 0.2143 - loss: 2.2887 - val_accuracy: 0.3000 - val_loss: 1.9584
Epoch 2/3
5/5 ━━━━━━━━━━━━━━━━━━━━ 13s 3s/step - accuracy: 0.1429 - loss: 2.3703 - val_accuracy: 0.2000 - val_loss: 1.6288
Epoch 3/3
5/5 ━━━━━━━━━━━━━━━━━━━━ 13s 3s/step - accuracy: 0.1929 - loss: 2.1920 - val_accuracy: 0.2000 - val_loss: 1.6816


In [31]:
# -----------------------------
# VALUTAZIONE FINALE SU TEST SET
# -----------------------------

test_loss, test_accuracy = model.evaluate(test_ds)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.2667 - loss: 1.6670
Test Loss: 1.6670
Test Accuracy: 0.2667
